## 01 — GeoJSON Structure

In the previous lesson we loaded plain JSON and navigated it as Python dicts and lists.

**GeoJSON** is still just JSON — the same rules apply. The difference is that GeoJSON follows a specific convention about what the keys mean. Once you learn that convention, every GeoJSON file you encounter will look familiar.

The example file for this lesson is `data/meteorites.geojson` — a dataset of meteorite landing sites around the world.

## The GeoJSON Shape

Every valid GeoJSON file is built from three nested layers:

```
FeatureCollection          ← the whole file
└── features: [ ... ]      ← a list of Feature objects
    └── Feature            ← one geographic thing
        ├── type: "Feature"
        ├── geometry       ← WHERE it is
        │   ├── type: "Point"
        │   └── coordinates: [lon, lat]
        └── properties     ← WHAT it is
            ├── name: "Aarhus"
            ├── mass: 720
            └── year: 1951
```

**Critical detail: coordinates are `[longitude, latitude]`, not `[lat, lon]`.**  
This is backwards from how most people say it out loud. It trips up everyone the first time.

## Loading the File

In [ ]:
import json
from pathlib import Path

path = Path("data/meteorites.geojson")
data = json.loads(path.read_text())

# inspect the top-level structure
print(type(data))           # <class 'dict'>
print(data.keys())          # dict_keys(['type', 'features'])
print(data["type"])         # FeatureCollection
print(len(data["features"])) # number of meteorite records

## Accessing a Single Feature

Unlike `latex_colors.json` where the top level was a list, here `data` is a dict and the list lives at `data["features"]`.

In [ ]:
feature = data["features"][0]

print(feature["type"])                        # Feature
print(feature["properties"]["name"])          # Aarhus
print(feature["properties"]["mass"])          # 720
print(feature["properties"]["year"])          # 1951
print(feature["geometry"]["type"])            # Point
print(feature["geometry"]["coordinates"])     # [10.23333, 56.18333]

## Unpacking Coordinates

`coordinates` is an array: `[longitude, latitude]`.  
Index `0` is longitude (east/west), index `1` is latitude (north/south).

In [ ]:
coords = feature["geometry"]["coordinates"]

lon = coords[0]   # 10.23333
lat = coords[1]   # 56.18333

print(f"lon: {lon},  lat: {lat}")

## Traversal — Loop Over All Features

In [ ]:
for feature in data["features"]:
    props = feature["properties"]
    coords = feature["geometry"]["coordinates"]
    print(f"{props['name']:<30} mass: {props['mass']:>10}  lon: {coords[0]}")

## Traversal — Look Up by Property Value

In [ ]:
match = next(
    (f for f in data["features"] if f["properties"].get("name") == "Acapulco"),
    None
)

if match:
    print(match["properties"])
    print(match["geometry"]["coordinates"])

## Traversal — Collect One Field Across All Features

In [ ]:
features = data["features"]

# all names
names = [f["properties"]["name"] for f in features]

# all (lon, lat) pairs
points = [f["geometry"]["coordinates"] for f in features]

# all masses — using .get() with a default since a field might be missing
masses = [f["properties"].get("mass", 0) for f in features]

print(names[:5])
print(points[:5])
print(masses[:5])

## GeoJSON vs Plain JSON — Key Differences

| | Plain JSON (`latex_colors.json`) | GeoJSON (`meteorites.geojson`) |
|---|---|---|
| Top-level type | array `[...]` | object `{...}` |
| How to get the list | `colors` directly | `data["features"]` |
| What each item is | an arbitrary object | always a `Feature` |
| Location data | none | `geometry.coordinates` |
| Attribute data | all keys at top level | lives inside `properties` |
| Coordinate order | n/a | `[lon, lat]` — not `[lat, lon]` |

The traversal pattern is the same — the structure just has a predictable shape every time.

## Exercise A

How many meteorites in this dataset have a recorded mass (non-`None`, non-zero)?

Use `.get("mass")` to safely access the field — some features may not have it.

In [ ]:
with_mass = [
    f for f in meteorites["features"]
    if f["properties"].get("mass") not in (None, 0)
]
print("Meteorites with recorded mass:", len(with_mass))


## Exercise B

Find all meteorites that fell **after the year 2000**. Print their names and years.

Use `.get("year")` with a default since some entries may be missing that field.

In [ ]:
after_2000 = [
    f for f in meteorites["features"]
    if (f["properties"].get("year") or 0) > 2000
]
print("Meteorites after 2000:", len(after_2000))
for feature in after_2000[:10]:
    print(feature["properties"]["name"], feature["properties"]["year"])


## Exercise C

Find the **3 northernmost** meteorites — those with the highest latitude. Print their names and latitudes, sorted from north to south.

Hint: latitude is `coordinates[1]`.

In [ ]:
northernmost = sorted(
    [f for f in meteorites["features"] if f.get("geometry")],
    key=lambda f: f["geometry"]["coordinates"][1],
    reverse=True,
)[:3]
for feature in northernmost:
    lon, lat = feature["geometry"]["coordinates"]
    print(feature["properties"]["name"], lat, lon)


```python
abee = next(f for f in meteorites["features"] if f["properties"].get("name") == "Abee")
print(abee["geometry"]["coordinates"][1], abee["properties"].get("year"))
```


## Next

In [02 — Feature Collections](./02-Feature_Collections.ipynb), we go beyond reading and start building and modifying GeoJSON collections from scratch.